# Files Required for Yolo Implementation

In [5]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

In [8]:
net = cv2.dnn.readNet('yolov3-tiny.weights','yolov3-tiny.cfg')
classes = []
with open('coco.names','r') as f:
    classes = [line.strip() for line in f.readlines()]
    
layers_name = net.getLayerNames()

In [9]:
output_layer = [layers_name[i[0] - 1] for i in net.getUnconnectedOutLayers()]
cap = cv2.VideoCapture(0)   


In [10]:
#define the capture video frame dimension
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))

# Send image until and unless camera is open
while (cap.isOpened()):
    ret,frame = cap.read() # ret(_) means capture unwanted values,where frame reads the image frame
    if ret == True:
        image = frame
        height,width,depth = image.shape
        #binary large object
        
        blob = cv2.dnn.blobFromImage(image,0.00392,(416,416),(0,0,0),True, crop = False) #0.00392 = scale factor : 0,0,0 = means of channels  
        net.setInput(blob)
        output = net.forward(output_layer)

        class_ids = []
        boxes = []
        confidences = []

        for out in output:
            for det in out:
                scores = det[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                if confidence > 0.6:
                    cx = int(det[0] * width) 
                    cy = int(det[1] * height)

                    w = int(det[2] * width)
                    h = int(det[3] * height)

                    x,y = int(cx-w/2),int(cy-h/2)
                    boxes.append([x,y,w,h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)

        n_det = len(boxes)
        indexes = cv2.dnn.NMSBoxes(boxes,confidences,0.5,0.3)

        for i in range(n_det):
            if i in indexes:
                x,y,w,h = boxes[i]
                label = classes[class_ids[i]]
                cv2.rectangle(image,(x,y),(x+w,y+h),(0,255,0),2)
                cv2.putText(image,label,(x,y+30),cv2.FONT_HERSHEY_PLAIN,5,(0,0,255),2)
            
                cv2.imshow('Yolo Video Detector',image)
                if cv2.waitKey(25) & 0xff == ord('q'):
                    break
                
        else:
            break
cap.release()
# output.release()
cv2.waitKey(0)
cv2.destroyAllWindows()